# Week 9: Unit Testing — Testing Your Classes — PHASE 4: Verifying & Swapping Components

*Object Oriented Programming . 3 Hours . Dr. Arif Solmaz*

---

## Learning Objectives

By the end of this week, you will be able to:

| # | Objective |
|---|----------|
| 1 | Explain why testing is important in software development |
| 2 | Use `assert` statements to verify code behavior |
| 3 | Write basic test cases using Python's `unittest` module |
| 4 | Test class methods systematically |
| 5 | Identify and test edge cases |
| 6 | Apply test-first thinking to your designs |

---

## 🎯 Core Mastery Connection

Each component needs independent verification. Unit tests prove a component works correctly before you plug it into a larger system. If every component is tested in isolation, you can compose them with confidence — knowing that failures come from the composition logic, not from broken parts.

---
## 📦 Setup

Run this cell first to load the required packages.

In [ ]:
import math
import unittest

## Part 1: Why Test?

Imagine you are building a controller for a robotic arm. The arm picks up objects from a conveyor belt and places them in boxes. You write the code, upload it, and the arm starts working.

But one day, the arm drops a heavy part because your code did not handle objects heavier than 5 kg. This bug could have been caught **before** deployment if you had written a test.

**Testing** means writing small programs that check whether your code works correctly.

### Why do engineers test code?

| Reason | Example |
|--------|---------|
| Catch bugs early | Find a math error before running on hardware |
| Save time | Automated tests run in seconds vs. manual checking |
| Safe changes | Modify code and verify nothing broke |
| Documentation | Tests show how code is supposed to be used |

**Figure 9.1** — The testing workflow:

```
Write Code  -->  Write Tests  -->  Run Tests  -->  Fix Bugs  -->  Repeat
```

> **Key Idea:** If you cannot test it, you cannot trust it.

---

## Part 2: Simple Assert Statements

The simplest way to test in Python is the `assert` statement. It checks if a condition is `True`. If the condition is `False`, Python raises an `AssertionError`.

```python
assert condition, "Error message if condition is False"
```

Think of `assert` as a self-check: "I expect this to be true. If it's not, something is wrong."

**Figure 9.1** — Testing with assertions

In [ ]:
# A simple function to convert Celsius to Fahrenheit
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

# Testing with assert statements
assert celsius_to_fahrenheit(0) == 32, "Freezing point should be 32F"
assert celsius_to_fahrenheit(100) == 212, "Boiling point should be 212F"
assert celsius_to_fahrenheit(-40) == -40, "-40 is the same in both scales"

print("All temperature conversion tests passed!")

**Figure 9.2** — with class implementation

In [ ]:
# Let's test a class with assert statements
class Motor:
    """A simple DC motor model."""
    def __init__(self, max_rpm):
        self.max_rpm = max_rpm
        self.current_rpm = 0

    def set_speed(self, rpm):
        """Set motor speed, clamped to max_rpm."""
        if rpm < 0:
            self.current_rpm = 0
        elif rpm > self.max_rpm:
            self.current_rpm = self.max_rpm
        else:
            self.current_rpm = rpm

    def stop(self):
        """Stop the motor."""
        self.current_rpm = 0


# Test the Motor class
m = Motor(3000)
assert m.current_rpm == 0, "Motor should start at 0 RPM"

m.set_speed(1500)
assert m.current_rpm == 1500, "Motor should be at 1500 RPM"

m.set_speed(5000)
assert m.current_rpm == 3000, "Motor should be clamped to max 3000 RPM"

m.set_speed(-100)
assert m.current_rpm == 0, "Negative RPM should be clamped to 0"

m.set_speed(2000)
m.stop()
assert m.current_rpm == 0, "Stop should set RPM to 0"

print("All Motor tests passed!")

### What happens when an assert fails?

Let's see:

**Figure 9.3** — Testing with assertions

In [ ]:
# Uncomment the line below to see what a failed assert looks like
# assert 2 + 2 == 5, "Math is broken!"

> **Tip:** Assert statements are great for quick checks, but for larger projects we need something more organized.

---

## Part 3: Introduction to unittest

Python has a built-in module called `unittest` that provides a structured way to write and run tests.

### Key concepts:

| Concept | Description |
|---------|-------------|
| **Test Case** | A class that contains test methods |
| **Test Method** | A method that starts with `test_` |
| **Assertion** | A check inside a test method |
| **Test Runner** | The system that runs all tests |

**Figure 9.2** — unittest structure:

```
unittest.TestCase  (base class)
    └── YourTestClass
            ├── test_method_1()
            ├── test_method_2()
            └── test_method_3()
```

In [ ]:
import unittest

# The class we want to test
class Resistor:
    """A simple resistor model."""
    def __init__(self, resistance):
        if resistance <= 0:
            raise ValueError("Resistance must be positive")
        self.resistance = resistance  # Ohms

    def voltage(self, current):
        """Calculate voltage using Ohm's law: V = I * R"""
        return current * self.resistance

    def current(self, voltage):
        """Calculate current using Ohm's law: I = V / R"""
        return voltage / self.resistance


# The test class
class TestResistor(unittest.TestCase):

    def test_voltage_calculation(self):
        r = Resistor(100)  # 100 Ohm resistor
        self.assertEqual(r.voltage(0.5), 50)  # 0.5A * 100 Ohm = 50V

    def test_current_calculation(self):
        r = Resistor(200)  # 200 Ohm resistor
        self.assertEqual(r.current(10), 0.05)  # 10V / 200 Ohm = 0.05A

    def test_zero_current(self):
        r = Resistor(100)
        self.assertEqual(r.voltage(0), 0)  # No current = no voltage


# Run tests in notebook (special syntax for Colab/Jupyter)
unittest.main(argv=[''], exit=False, verbosity=2)

### Common unittest assertions

| Method | Checks that |
|--------|------------|
| `assertEqual(a, b)` | `a == b` |
| `assertNotEqual(a, b)` | `a != b` |
| `assertTrue(x)` | `x` is `True` |
| `assertFalse(x)` | `x` is `False` |
| `assertAlmostEqual(a, b)` | `a ≈ b` (for floats) |
| `assertRaises(Error)` | An exception is raised |

> **Note:** Use `assertAlmostEqual` when comparing floating-point numbers, because floats can have tiny rounding errors.

---

## Part 4: Writing Your First Test Case

Let's write a test case step by step. We will test a `Sensor` class that reads temperature values.

**Steps to write a test:**
1. Create a class that inherits from `unittest.TestCase`
2. Write methods that start with `test_`
3. Use assertion methods to check expected results
4. Run the tests

**Figure 9.4** — TemperatureSensor class implementation

In [ ]:
import unittest

class TemperatureSensor:
    """A temperature sensor that stores readings."""
    def __init__(self, unit="C"):
        self.unit = unit
        self.readings = []

    def add_reading(self, value):
        """Add a temperature reading."""
        self.readings.append(value)

    def average(self):
        """Return the average of all readings."""
        if len(self.readings) == 0:
            return 0
        return sum(self.readings) / len(self.readings)

    def max_reading(self):
        """Return the highest reading."""
        if len(self.readings) == 0:
            return None
        return max(self.readings)

    def count(self):
        """Return the number of readings."""
        return len(self.readings)


# Step-by-step test case
class TestTemperatureSensor(unittest.TestCase):

    def test_new_sensor_has_no_readings(self):
        # Step 1: Create the object
        sensor = TemperatureSensor()
        # Step 2: Check the expected result
        self.assertEqual(sensor.count(), 0)

    def test_add_one_reading(self):
        sensor = TemperatureSensor()
        sensor.add_reading(25.0)
        self.assertEqual(sensor.count(), 1)

    def test_average_of_readings(self):
        sensor = TemperatureSensor()
        sensor.add_reading(20.0)
        sensor.add_reading(30.0)
        self.assertAlmostEqual(sensor.average(), 25.0)

    def test_max_reading(self):
        sensor = TemperatureSensor()
        sensor.add_reading(18.0)
        sensor.add_reading(22.5)
        sensor.add_reading(19.3)
        self.assertEqual(sensor.max_reading(), 22.5)


unittest.main(argv=[''], exit=False, verbosity=2)

---

## Part 5: Testing Class Methods

When testing a class, think about **what each method should do** and write a test for each behavior.

**Figure 9.3** — What to test for each method:

```
Method: set_speed(rpm)
    ├── Test: Normal value (e.g., 1500)
    ├── Test: Boundary value (e.g., 0, max_rpm)
    ├── Test: Too high value (e.g., max_rpm + 1)
    └── Test: Negative value (e.g., -100)
```

In [ ]:
import unittest

class ServoMotor:
    """A servo motor that moves to an angle between 0 and 180 degrees."""
    def __init__(self):
        self.angle = 0

    def move_to(self, degrees):
        """Move servo to a given angle."""
        if degrees < 0:
            self.angle = 0
        elif degrees > 180:
            self.angle = 180
        else:
            self.angle = degrees

    def center(self):
        """Move servo to center position (90 degrees)."""
        self.angle = 90

    def is_at_limit(self):
        """Check if servo is at 0 or 180."""
        return self.angle == 0 or self.angle == 180


class TestServoMotor(unittest.TestCase):

    def test_initial_angle(self):
        servo = ServoMotor()
        self.assertEqual(servo.angle, 0)

    def test_move_to_normal_angle(self):
        servo = ServoMotor()
        servo.move_to(45)
        self.assertEqual(servo.angle, 45)

    def test_move_to_max_angle(self):
        servo = ServoMotor()
        servo.move_to(180)
        self.assertEqual(servo.angle, 180)

    def test_move_beyond_max(self):
        servo = ServoMotor()
        servo.move_to(200)
        self.assertEqual(servo.angle, 180)

    def test_move_negative(self):
        servo = ServoMotor()
        servo.move_to(-10)
        self.assertEqual(servo.angle, 0)

    def test_center(self):
        servo = ServoMotor()
        servo.move_to(45)
        servo.center()
        self.assertEqual(servo.angle, 90)

    def test_is_at_limit_true(self):
        servo = ServoMotor()
        self.assertTrue(servo.is_at_limit())  # At 0 degrees

    def test_is_at_limit_false(self):
        servo = ServoMotor()
        servo.move_to(90)
        self.assertFalse(servo.is_at_limit())


unittest.main(argv=[''], exit=False, verbosity=2)

---

## Part 6: Edge Cases — What Could Go Wrong?

**Edge cases** are unusual or extreme inputs that might cause your code to break. Good engineers always think about what could go wrong.

### Common edge cases to test:

| Edge Case | Example |
|-----------|--------|
| Empty input | An empty list of sensor readings |
| Zero value | Speed = 0, weight = 0 |
| Negative values | Negative temperature, negative distance |
| Very large values | Speed = 999999 |
| Invalid types | Passing a string instead of a number |
| Boundary values | Exactly at min or max limit |

**Figure 9.5** — Tank class implementation

In [ ]:
import unittest

class Tank:
    """A water tank with a capacity limit."""
    def __init__(self, capacity):
        self.capacity = capacity  # liters
        self.level = 0            # current water level

    def fill(self, amount):
        """Add water to the tank."""
        if amount < 0:
            raise ValueError("Cannot fill negative amount")
        self.level = min(self.level + amount, self.capacity)

    def drain(self, amount):
        """Remove water from the tank."""
        if amount < 0:
            raise ValueError("Cannot drain negative amount")
        self.level = max(self.level - amount, 0)

    def is_full(self):
        return self.level == self.capacity

    def is_empty(self):
        return self.level == 0


class TestTankEdgeCases(unittest.TestCase):

    def test_new_tank_is_empty(self):
        tank = Tank(100)
        self.assertTrue(tank.is_empty())

    def test_fill_to_capacity(self):
        tank = Tank(100)
        tank.fill(100)
        self.assertTrue(tank.is_full())

    def test_overfill(self):
        # Edge case: filling more than capacity
        tank = Tank(100)
        tank.fill(150)
        self.assertEqual(tank.level, 100)  # Should not exceed capacity

    def test_drain_more_than_level(self):
        # Edge case: draining more than available
        tank = Tank(100)
        tank.fill(30)
        tank.drain(50)
        self.assertEqual(tank.level, 0)  # Should not go below 0

    def test_fill_zero(self):
        # Edge case: filling zero amount
        tank = Tank(100)
        tank.fill(0)
        self.assertEqual(tank.level, 0)

    def test_negative_fill_raises_error(self):
        # Edge case: negative amount should raise error
        tank = Tank(100)
        with self.assertRaises(ValueError):
            tank.fill(-10)

    def test_negative_drain_raises_error(self):
        tank = Tank(100)
        with self.assertRaises(ValueError):
            tank.drain(-10)


unittest.main(argv=[''], exit=False, verbosity=2)

> **Tip:** When you write a class, ask yourself: "What is the weirdest thing someone could pass to this method?" Then write a test for it.

---

## Part 7: Organizing Your Tests

As your project grows, you will have many tests. Here are some tips for keeping them organized:

### Naming conventions

| What | Convention |
|------|----------|
| Test class name | `TestClassName` (e.g., `TestMotor`) |
| Test method name | `test_what_it_checks` (e.g., `test_speed_is_clamped`) |

### The setUp method

If many tests need the same setup, use the `setUp` method. It runs **before each test**.

**Figure 9.4** — setUp runs before every test method:

```
setUp()  -->  test_method_1()
setUp()  -->  test_method_2()
setUp()  -->  test_method_3()
```

In [ ]:
import unittest

class ConveyorBelt:
    """A conveyor belt that moves items."""
    def __init__(self, speed):
        self.speed = speed  # meters per second
        self.items = []

    def add_item(self, item):
        self.items.append(item)

    def remove_item(self):
        if len(self.items) == 0:
            return None
        return self.items.pop(0)  # Remove first item (FIFO)

    def item_count(self):
        return len(self.items)


class TestConveyorBelt(unittest.TestCase):

    def setUp(self):
        """This runs before EACH test method."""
        self.belt = ConveyorBelt(speed=1.5)
        self.belt.add_item("Part A")
        self.belt.add_item("Part B")

    def test_initial_item_count(self):
        # setUp already added 2 items
        self.assertEqual(self.belt.item_count(), 2)

    def test_remove_returns_first_item(self):
        item = self.belt.remove_item()
        self.assertEqual(item, "Part A")

    def test_remove_decreases_count(self):
        self.belt.remove_item()
        self.assertEqual(self.belt.item_count(), 1)

    def test_remove_from_empty(self):
        self.belt.remove_item()  # Remove Part A
        self.belt.remove_item()  # Remove Part B
        result = self.belt.remove_item()  # Empty!
        self.assertIsNone(result)


unittest.main(argv=[''], exit=False, verbosity=2)

### Test-first thinking

Some engineers write the **tests before** the code. This is called **Test-Driven Development (TDD)**.

The idea is simple:
1. Write a test for what you want the code to do
2. Run the test (it will fail because the code doesn't exist yet)
3. Write the minimum code to make the test pass
4. Repeat

You do not need to follow TDD strictly, but thinking about tests **early** helps you write better code.

---

## Exercises

> **Composition lens:** Every test you write verifies that a component meets its contract. When you test a `Motor`, `Tank`, or `Gripper` in isolation, you are ensuring it is a reliable building block — ready to be composed into larger systems without surprises.

Complete the following exercises in the code cells below. Each exercise asks you to write tests or classes with tests.

### Exercise 1: Assert Statements for a Function (Easy)

Write a function `km_to_miles(km)` that converts kilometers to miles (1 km = 0.621371 miles). Then write at least 3 assert statements to test it.

<details>
<summary>💡 Hint</summary>

Test with known values: 0 km = 0 miles, 1 km ≈ 0.621371 miles. Use `round()` for float comparison or check with a small tolerance.
</details>

In [ ]:
# ✏️ [EX1] Write your km_to_miles function and assert tests here


### Exercise 2: Test a Battery Class (Easy)

Given the `Battery` class below, write a `TestBattery` class with at least 4 test methods.

```python
class Battery:
    def __init__(self, capacity):
        self.capacity = capacity
        self.charge = capacity  # Starts fully charged
    
    def use(self, amount):
        self.charge = max(0, self.charge - amount)
    
    def recharge(self):
        self.charge = self.capacity
    
    def percentage(self):
        return (self.charge / self.capacity) * 100
```

<details>
<summary>💡 Hint</summary>

Test: starts fully charged, use reduces charge, cannot go below 0, recharge restores to full, percentage calculation.
</details>

In [ ]:
# ✏️ [EX2] Write the Battery class and TestBattery test class here
import unittest


### Exercise 3: Edge Case Tests (Easy)

Write a class `PressureSensor` with a method `add_reading(value)` that only accepts values between 0 and 1000 (inclusive). If the value is outside this range, raise a `ValueError`. Write tests for normal values AND edge cases.

<details>
<summary>💡 Hint</summary>

Edge cases to test: value = 0, value = 1000, value = -1, value = 1001. Use `assertRaises` for invalid values.
</details>

In [ ]:
# ✏️ [EX3] Write PressureSensor class and its tests here
import unittest


### Exercise 4: Test a Counter Class (Easy)

Create a `StepCounter` class that counts steps. It should have:
- `add_steps(n)` — adds n steps (only positive values)
- `total()` — returns total steps
- `reset()` — resets to 0

Write the class AND at least 4 tests.

<details>
<summary>💡 Hint</summary>

Test that total starts at 0, adding steps increases total, reset works, and negative steps are rejected.
</details>

In [ ]:
# ✏️ [EX4] Write StepCounter class and tests here
import unittest


### Exercise 5: setUp Method (Medium)

Rewrite your `TestBattery` tests from Exercise 2, but this time use a `setUp` method to create the Battery object before each test.

<details>
<summary>💡 Hint</summary>

In `setUp`, create `self.battery = Battery(100)`. Then each test method can use `self.battery` directly.
</details>

In [ ]:
# ✏️ [EX5] Rewrite TestBattery with setUp method
import unittest


### Exercise 6: Test assertAlmostEqual (Medium)

Write a class `CircleArea` with a method `area(radius)` that returns the area of a circle (πr²). Write tests using `assertAlmostEqual` to handle floating-point precision.

<details>
<summary>💡 Hint</summary>

Use `import math` and `math.pi`. For radius=1, area should be approximately 3.14159. Use `assertAlmostEqual(result, expected, places=4)`.
</details>

In [ ]:
# ✏️ [EX6] Write CircleArea class and tests with assertAlmostEqual
import unittest
import math


### Exercise 7: Test-First Thinking (Medium)

Write the **tests first** for a `Thermostat` class, then implement the class to make the tests pass.

The Thermostat should have:
- `__init__(target_temp)` — sets the target temperature
- `check(current_temp)` — returns `"HEAT"` if current is below target, `"COOL"` if above, `"OK"` if equal

<details>
<summary>💡 Hint</summary>

Write three test methods first: one for each return value ("HEAT", "COOL", "OK"). Then write the class to pass them.
</details>

In [ ]:
# ✏️ [EX7] Write tests FIRST, then implement Thermostat
import unittest


### Exercise 8: Test a Gripper Class (Medium)

Create a `Gripper` class for a robotic gripper:
- `open()` — opens the gripper
- `close()` — closes the gripper
- `is_open()` — returns True/False
- `grab(weight)` — grabs an object if closed and weight <= 10 kg. Returns True if successful.

Write the class and at least 5 tests.

<details>
<summary>💡 Hint</summary>

Test: starts open, close/open works, cannot grab when open, can grab within weight limit, cannot grab over weight limit.
</details>

In [ ]:
# ✏️ [EX8] Write Gripper class and tests
import unittest


### Exercise 9: Test a Queue Class (Medium)

Create a `TaskQueue` class that manages tasks in FIFO order:
- `add_task(task_name)` — adds a task
- `next_task()` — removes and returns the first task (or None if empty)
- `size()` — returns number of tasks
- `is_empty()` — returns True if no tasks

Write the class and at least 5 tests including edge cases.

<details>
<summary>💡 Hint</summary>

Test: starts empty, adding increases size, next_task returns first added, next_task on empty returns None, is_empty after removing all.
</details>

In [ ]:
# ✏️ [EX9] Write TaskQueue class and tests
import unittest


### Exercise 10: Find the Bug (Challenge)

The following `SpeedController` class has a bug. Write tests to find it, then fix the class.

```python
class SpeedController:
    def __init__(self, max_speed):
        self.max_speed = max_speed
        self.speed = 0
    
    def accelerate(self, amount):
        self.speed += amount
    
    def brake(self, amount):
        self.speed -= amount
```

Expected behavior: speed should never go above `max_speed` or below 0.

<details>
<summary>💡 Hint</summary>

The bug is that `accelerate` and `brake` don't clamp the values. Write tests that expose this, then fix the methods using `min()` and `max()`.
</details>

In [ ]:
# ✏️ [EX10] Write tests to find the bug, then fix SpeedController
import unittest


### Exercise 11: Comprehensive Test Suite (Challenge)

Write a `LEDStrip` class with:
- `__init__(num_leds)` — creates a strip with `num_leds` LEDs, all off (False)
- `turn_on(index)` — turns on the LED at given index
- `turn_off(index)` — turns off the LED at given index
- `all_on()` — turns all LEDs on
- `all_off()` — turns all LEDs off
- `count_on()` — returns number of LEDs that are on

Write at least 6 tests covering normal usage and edge cases.

<details>
<summary>💡 Hint</summary>

Use a list of booleans to store LED states. Edge cases: index out of range, count_on with no LEDs on, all_on then all_off.
</details>

In [ ]:
# ✏️ [EX11] Write LEDStrip class and comprehensive tests
import unittest


### Exercise 12 (Studio): Write Test Cells and Edge-Case Set

Create a `RobotArm` class with:
- `__init__(reach)` — reach in centimeters
- `move_to(x, y)` — moves to position if within reach (distance from origin <= reach)
- `current_position()` — returns (x, y) tuple
- `distance_from_origin()` — returns distance from (0, 0)

Write the class, then write a full test suite with:
- At least 3 normal tests
- At least 3 edge-case tests
- Use `setUp` method

<details>
<summary>💡 Hint</summary>

Distance formula: `sqrt(x**2 + y**2)`. Edge cases: move to origin, move exactly at reach, move beyond reach, negative coordinates.
</details>

In [ ]:
# ✏️ [EX12] Studio: RobotArm class with full test suite
import unittest
import math


---

## 🌉 Bridge to Next Week

Now that you know how to test your classes, you can confidently build more complex systems. Next week, we will learn about **Design Patterns** — proven solutions to common programming problems. Specifically, we will look at the **Strategy Pattern**, which lets you swap different behaviors at runtime. Testing will help us verify that our patterns work correctly!

---

## 📮 Submission

Follow the two steps below to submit your work.

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 1: Fill in your info below, then run this cell#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━STUDENT_ID    = ""     # e.g. "2024001234"STUDENT_NAME  = ""     # e.g. "Ahmet Yılmaz"STUDENT_EMAIL = ""     # e.g. "ahmet.yilmaz@istun.edu.tr"CLASS_CODE    = ""     # code given in class#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# Don't change anything below this line#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import re as _re_errors = []if not _re.match(r"^\d{6,12}$", STUDENT_ID):    _errors.append("❌ Student ID must be 6-12 digits")if len(STUDENT_NAME.strip().split()) < 2:    _errors.append("❌ Enter first and last name")if not STUDENT_EMAIL.strip().lower().endswith("@istun.edu.tr") or len(STUDENT_EMAIL.strip()) < 16:    _errors.append("❌ Use your @istun.edu.tr email")if len(CLASS_CODE.strip()) < 4:    _errors.append("❌ Invalid class code")if _errors:    for _e in _errors:        print(_e)    print("\n⚠️  Fix the errors above and run this cell again.")else:    print(f"✅ Info OK — {STUDENT_NAME} ({STUDENT_ID})")    print(f"   {STUDENT_EMAIL}")    print(f"\n👉 Now run the NEXT cell to submit.")

In [ ]:
#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━# 📮 STEP 2: Run this cell to submit#━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━import json, re, os, urllib.requestWEEK = "Week_09"URL  = "https://script.google.com/macros/s/AKfycbyf1D3HGSAX4MoIhNlAuWlGrFyyvbM5MIv7ZsLxrVDlATUihrRGEAaibvIZYlCfd8Me/exec"try:    _sid = STUDENT_ID.strip()    _sname = STUDENT_NAME.strip()    _semail = STUDENT_EMAIL.strip().lower()    _scode = CLASS_CODE.strip().upper()except NameError:    raise SystemExit("❌ Run the cell above first to set your info!")if not _sid or not _sname or not _semail or not _scode:    raise SystemExit("❌ Run the cell above first — some fields are empty.")_answers = {}try:    _ipy = get_ipython()    _hist = _ipy.history_manager.get_range(output=False)    for _sess, _line, _src in _hist:        _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)        if _m:            _ex_id = "ex" + _m.group(1)            _lines = _src.split("\n")            _clean = "\n".join(_lines[1:]).strip()            _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}except Exception:    passif not _answers:    try:        for _src in In:            if not _src: continue            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}    except NameError:        passif not _answers:    _nb_path = None    try:        _nb_path = __vsc_ipynb_file__    except NameError:        _candidates = [f for f in os.listdir(".") if f.endswith(".ipynb") and WEEK in f]        if len(_candidates) == 1: _nb_path = _candidates[0]    if _nb_path and os.path.exists(str(_nb_path)):        with open(str(_nb_path), "r", encoding="utf-8") as _f:            _nb = json.load(_f)        for _cell in _nb["cells"]:            if _cell["cell_type"] != "code": continue            _src = "".join(_cell["source"]) if isinstance(_cell["source"], list) else _cell["source"]            _m = re.match(r"#\s*✏️\s*\[EX(\w+)\]", _src)            if _m:                _ex_id = "ex" + _m.group(1)                _lines = _src.split("\n")                _clean = "\n".join(_lines[1:]).strip()                _answers[_ex_id] = {"code": _clean, "modified": len(_clean) > 5}print(f"📝 Found {len(_answers)} exercise(s): {', '.join(sorted(_answers.keys()))}")if not _answers:    print("\n⚠️  No exercise answers found!")    print("Make sure you RAN all exercise cells before submitting.")    raise SystemExit()_data = json.dumps({"week": WEEK, "studentId": _sid, "studentName": _sname, "studentEmail": _semail, "classCode": _scode, "source": "oop-notebook", "timeOnPage": 0, "answers": _answers}).encode("utf-8")print("📡 Submitting...")try:    _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")    _resp = urllib.request.urlopen(_req, timeout=30)    _result = json.loads(_resp.read().decode())    if _result.get("success"):        print(f"\n✅ {_result['message']}")        print("📧 Check your email for confirmation.")    else:        print(f"\n❌ {_result.get('message', 'Submission failed')}")except Exception as _e:    try:        _req = urllib.request.Request(URL, data=_data, headers={"Content-Type": "text/plain"}, method="POST")        urllib.request.urlopen(_req, timeout=10)    except: pass    print(f"\n⚠️  Request sent — check your email for confirmation.")    print(f"(If no email arrives, try again or contact your instructor)")